# 03. LoRA 파인튜닝 실습: 같은 "고양이 비서"를 LoRA로 만들기

2단계에서는 Qwen2.5-0.5B의 **모든 가중치(약 5억 개)** 를 학습했습니다(Full 파인튜닝).
이번에는 같은 데이터, 같은 과제를 **LoRA(Low-Rank Adaptation)** 로 학습합니다. 원래 가중치는 그대로 얼려 두고, 옆에 붙인 **아주 작은 행렬만** 학습하는 방법입니다.

**이 노트북에서 배우는 것**

1. LoRA의 원리: `W + B·A`, rank `r`, `alpha`, 어디에 붙이는가(`target_modules`)
2. 순수 PyTorch로 LoRA 레이어를 직접 만들어 보기
3. PEFT 라이브러리로 Qwen2.5-0.5B에 LoRA 붙이고 `Trainer`로 학습하기
4. 어댑터를 켜고 끄며 학습 전후 비교하기 (베이스 모델을 다시 불러올 필요 없음)
5. 어댑터만 저장하기 (수 GB → 수십 MB)
6. Full 파인튜닝과 학습 파라미터 수, 메모리, 저장 크기, 결과 비교
7. (선택) 12GB GPU에서 Full로는 못 하는 **1.5B 모델을 LoRA로 학습**하기

**실행 환경**

| 환경 | 예상 시간 | 필요 메모리 |
|---|---|---|
| RTX 3060 12GB | 약 1~2분 (학습) | VRAM 약 3~5GB |
| CPU만 있는 경우 | 약 5~20분 (스텝 수를 줄인 설정) | RAM 8GB 이상 권장 |

2단계를 먼저 해 보고 오면 비교가 쉽습니다. 2단계와 겹치는 코드(데이터 준비, 프롬프트 형식)는 설명을 줄이고 그대로 가져왔습니다.

> 💡 셀을 위에서부터 순서대로 실행하세요 (`Shift + Enter`). GPU 메모리가 꼬이면 커널을 재시작하고 처음부터 다시 실행하면 됩니다.

## 1. LoRA의 아이디어

Full 파인튜닝은 가중치 행렬 `W`(예: 896×896) 전체를 업데이트합니다. 학습 후의 가중치를 `W + ΔW`라고 쓰면, `ΔW`도 896×896 = 약 80만 개의 숫자입니다.

LoRA의 관찰은 이렇습니다. **파인튜닝으로 생기는 변화 `ΔW`는 "단순한" 변화라서, 작은 행렬 두 개의 곱으로 충분히 표현된다.**

```
         원래 가중치 (얼림)        LoRA (학습)
y = x · Wᵀ          +   (x · Aᵀ · Bᵀ) × (alpha / r)

W : 896 × 896  (802,816개, 학습 안 함)
A :   r × 896  (r=16이면 14,336개)
B : 896 ×   r  (r=16이면 14,336개)   → 합쳐서 28,672개 = W의 3.6%
```

| 용어 | 뜻 |
|---|---|
| **rank `r`** | A와 B 사이의 "좁은 통로" 크기. 클수록 표현력이 커지고 파라미터도 늘어남. 보통 8~64 |
| **`lora_alpha`** | LoRA 출력에 곱하는 배율의 분자. 실제 배율은 `alpha / r`. 보통 `r`의 1~2배 |
| **`target_modules`** | LoRA를 붙일 Linear 레이어 이름. 어텐션(`q/k/v/o_proj`)만 붙이거나, MLP(`gate/up/down_proj`)까지 전부 붙임 |
| **`lora_dropout`** | LoRA 입력에 거는 dropout. 데이터가 적을 때 과적합 방지 |

**왜 메모리가 크게 줄까?** 1단계에서 Full 파인튜닝 메모리를 계산했을 때, 가중치(4바이트)보다 **gradient(4바이트) + AdamW 상태(8바이트)** 가 훨씬 컸습니다.
LoRA는 학습하는 파라미터가 1~2%뿐이라 gradient와 옵티마이저 상태가 거의 사라집니다.
게다가 원래 가중치는 업데이트하지 않으니 fp32로 들고 있을 필요 없이 **bf16(2바이트)** 으로 올려도 됩니다.

## 2. 순수 PyTorch로 LoRA 레이어 만들어 보기

라이브러리를 쓰기 전에 원리를 코드로 확인합니다. Qwen2.5-0.5B의 어텐션 Linear 한 개(896→896) 크기로 만들어 봅니다.

핵심 포인트 두 가지:
- **`B`를 0으로 초기화**합니다. 그래서 학습 시작 시점에는 LoRA 출력이 0이고, 모델은 **원래 모델과 완전히 똑같이** 동작합니다.
- 원래 Linear의 파라미터는 `requires_grad = False`로 얼립니다.

In [ ]:
import json
import math
import os
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


class LoRALinear(nn.Module):
    """원래 Linear(W)는 얼리고, 옆에 작은 A, B 행렬을 붙인 레이어"""

    def __init__(self, base: nn.Linear, r=16, alpha=32):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                     # 원래 가중치는 학습 안 함
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # 0으로 시작!
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale


base_linear = nn.Linear(896, 896, bias=False)
lora_linear = LoRALinear(base_linear, r=16, alpha=32)

n_total = sum(p.numel() for p in lora_linear.parameters())
n_train = sum(p.numel() for p in lora_linear.parameters() if p.requires_grad)
print(f"원래 W 파라미터  : {base_linear.weight.numel():,}")
print(f"LoRA(A+B) 파라미터: {n_train:,}  ({n_train / base_linear.weight.numel():.1%})")

x = torch.randn(4, 896)
print("\n학습 전 출력이 원래 Linear와 같은가?", torch.allclose(lora_linear(x), base_linear(x)))

이제 "파인튜닝으로 생긴 변화"를 흉내 내 봅시다. 원래 가중치 `W`에 **rank 4짜리 변화** `ΔW`를 더한 목표 레이어를 만들고, LoRA의 `A`, `B`만 학습해서 그 목표를 따라가게 합니다.
`W`는 전혀 건드리지 않는데도 오차가 거의 0까지 줄어드는 것을 볼 수 있습니다.

In [ ]:
# 목표: W + ΔW  (ΔW는 rank 4짜리 작은 변화)
delta_W = (torch.randn(896, 4) @ torch.randn(4, 896)) * 0.01
target_W = base_linear.weight.detach() + delta_W
W_before = base_linear.weight.detach().clone()   # 원래 W를 복사해 두고 나중에 비교

optimizer = torch.optim.AdamW([lora_linear.A, lora_linear.B], lr=1e-2)
for step in range(301):
    x = torch.randn(64, 896)
    loss = F.mse_loss(lora_linear(x), x @ target_W.T)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f"step {step:3d} | loss {loss.item():.6f}")

print("\n원래 W는 그대로인가?", torch.equal(W_before, lora_linear.base.weight))

`W`는 그대로인데(`True`) `A`, `B`만으로 변화를 배웠습니다.

> 📝 실제 LLM 파인튜닝의 `ΔW`가 정확히 low-rank인 것은 아니지만, "말투 바꾸기"처럼 좁은 과제에서는 low-rank로 근사해도 Full 파인튜닝과 거의 같은 결과가 나온다는 것이 LoRA 논문(Hu et al., 2021)의 발견입니다.
> 반대로 새로운 언어나 대량의 지식을 넣는 것처럼 큰 변화가 필요하면 `r`을 키우거나 Full 파인튜닝이 더 나을 수 있습니다.

이제 이 일을 실제 모델의 수백 개 Linear에 자동으로 해 주는 **PEFT** 라이브러리를 써 봅시다.

## 3. 환경 확인과 실행 설정

2단계와 같은 방식으로 GPU/CPU 설정을 자동으로 고릅니다. 2단계와 다른 점은 표시해 두었습니다.

- **학습률이 10배 정도 큽니다** (`3e-5` → `2e-4`). LoRA는 0에서 시작하는 작은 행렬만 학습하므로 큰 학습률이 필요합니다. Full 파인튜닝에 이 학습률을 쓰면 모델이 망가집니다.
- **베이스 모델을 bf16으로 올립니다** (GPU에서). 얼린 가중치는 업데이트하지 않으니 fp32 원본이 필요 없습니다. 학습하는 LoRA 가중치는 PEFT가 자동으로 fp32로 유지합니다.

In [ ]:
import transformers
import datasets
import peft

print("PyTorch      :", torch.__version__)
print("transformers :", transformers.__version__)
print("datasets     :", datasets.__version__)
print("peft         :", peft.__version__)

if torch.cuda.is_available():
    device = "cuda"                      # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"                       # Apple Silicon Mac (M1~M4) GPU
else:
    device = "cpu"
print("사용할 장치  :", device)

if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU 이름     :", props.name)
    print(f"VRAM         : {props.total_memory / 1024**3:.1f} GB")
    print("bf16 지원    :", torch.cuda.is_bf16_supported())

transformers.set_seed(42)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"   # 2단계와 같은 베이스 모델

if device in ("cuda", "mps"):
    # Mac(mps)은 GPU 설정을 쓰되, 혼합 정밀도(bf16/fp16) 학습은 MPS에서 불안정할 수 있어 fp32로 학습한다
    use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
    config = dict(
        max_length=256,
        batch_size=4,
        grad_accum_steps=2,
        epochs=5,
        lr=2e-4,                 # ← Full(3e-5)보다 훨씬 큼
        bf16=use_bf16,
        fp16=device == "cuda" and not use_bf16,
        base_dtype=(torch.bfloat16 if use_bf16 else torch.float16) if device == "cuda" else torch.float32,   # ← NVIDIA GPU에서는 얼린 가중치를 16비트로 (Mac은 fp32)
        gradient_checkpointing=False,
    )
else:
    config = dict(
        max_length=128,
        batch_size=2,
        grad_accum_steps=4,
        epochs=2,
        lr=3e-4,
        bf16=False,
        fp16=False,
        base_dtype=torch.float32,  # CPU는 fp32
        gradient_checkpointing=False,
    )
    torch.set_num_threads(os.cpu_count())

# LoRA 설정
lora_settings = dict(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",      # 어텐션
                    "gate_proj", "up_proj", "down_proj"],        # MLP
)

for k, v in {**config, **lora_settings}.items():
    print(f"{k:24}: {v}")
print(f"{'effective batch size':24}: {config['batch_size'] * config['grad_accum_steps']}")

## 4. 토크나이저, 데이터, 프롬프트 형식 (2단계와 동일)

비교를 공정하게 하려고 2단계와 **똑같은 데이터, 프롬프트 형식, 토큰화, collator**를 씁니다. 자세한 설명은 2단계 노트북 4장을 참고하세요.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
raw = load_dataset(
    "json",
    data_files={
        "train": str(DATA_DIR / "cat_assistant_train.jsonl"),
        "eval": str(DATA_DIR / "cat_assistant_eval.jsonl"),
    },
)


def build_prompt(question):
    return f"### 질문:\n{question}\n\n### 답변:\n"


def tokenize_example(example):
    prompt_ids = tokenizer(build_prompt(example["question"]), add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(example["answer"], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    input_ids = (prompt_ids + answer_ids)[: config["max_length"]]
    labels = ([-100] * len(prompt_ids) + answer_ids)[: config["max_length"]]   # 답변만 loss
    return {"input_ids": input_ids, "labels": labels}


def collate(examples):
    max_len = max(len(ex["input_ids"]) for ex in examples)
    input_ids, labels, attention_mask = [], [], []
    for ex in examples:
        n_pad = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [tokenizer.pad_token_id] * n_pad)
        labels.append(ex["labels"] + [-100] * n_pad)
        attention_mask.append([1] * len(ex["input_ids"]) + [0] * n_pad)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attention_mask),
    }


tokenized = raw.map(tokenize_example, remove_columns=raw["train"].column_names)
print(tokenized)

In [ ]:
@torch.no_grad()
def ask(model, question, max_new_tokens=80):
    model.eval()
    inputs = tokenizer(build_prompt(question), return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


test_questions = [
    "고양이는 몇 시간 자?",
    "서울에서 부산까지 어떻게 가?",
    "LLM이 뭐야?",
    "오늘 저녁 메뉴 추천해 줘.",
]

## 5. 베이스 모델 불러오기

GPU에서는 `config["base_dtype"]`(bf16)으로 올립니다. 2단계(fp32)보다 모델이 차지하는 메모리가 절반입니다.

In [ ]:
from transformers import AutoModelForCausalLM

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device, dtype=config["base_dtype"])

print("가중치 dtype:", next(model.parameters()).dtype)
print(f"전체 파라미터: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
if device == "cuda":
    print(f"모델만 올렸을 때 VRAM 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB  (2단계 fp32는 약 2GB)")

LoRA를 붙일 Linear 레이어들이 실제로 어떤 이름인지 첫 번째 레이어를 들여다봅시다. `target_modules`에 적은 이름이 여기서 나옵니다.

In [ ]:
print(model.model.layers[0])

## 6. PEFT로 LoRA 붙이기

`LoraConfig`에 설정을 적고 `get_peft_model`을 부르면, `target_modules`에 해당하는 모든 Linear가 2장에서 만든 `LoRALinear` 같은 레이어로 바뀝니다.
원래 가중치는 자동으로 얼려집니다.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    bias="none",
    **lora_settings,
)

if config["gradient_checkpointing"]:
    model.config.use_cache = False

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"\n학습할 파라미터: {n_trainable / 1e6:.2f}M  ← Full 파인튜닝은 {(n_all - n_trainable) / 1e6:.0f}M 전부")
print("LoRA 가중치 dtype:", next(p for p in model.parameters() if p.requires_grad).dtype)

`q_proj`가 어떻게 바뀌었는지 봅시다. `base_layer`(원래 Linear)와 `lora_A`, `lora_B`가 보입니다. 2장에서 직접 만든 구조와 같습니다.

In [ ]:
print(model.base_model.model.model.layers[0].self_attn.q_proj)

### 학습 전 답변

LoRA의 `B`는 0으로 시작하므로, 지금 모델은 **베이스 모델과 똑같이** 답해야 합니다. 2단계의 "학습 전" 답변과 비교해 보세요.
(GPU에서는 bf16이라 아주 미세하게 다를 수 있습니다.)

In [ ]:
before_answers = {}
for q in test_questions:
    before_answers[q] = ask(model, q)
    print(f"Q: {q}\nA: {before_answers[q]}\n" + "-" * 60)

## 7. `Trainer`로 LoRA 학습

`Trainer` 사용법은 2단계와 똑같습니다. 모델만 PEFT 모델로 바뀌었을 뿐입니다. 옵티마이저는 `requires_grad=True`인 파라미터(LoRA)만 업데이트합니다.

In [ ]:
from transformers import Trainer, TrainingArguments

OUTPUT_DIR = Path("../outputs/03_lora") if Path("../data").exists() else Path("outputs/03_lora")

steps_per_epoch = math.ceil(len(tokenized["train"]) / (config["batch_size"] * config["grad_accum_steps"]))
total_steps = steps_per_epoch * config["epochs"]
print(f"epoch당 업데이트 횟수: {steps_per_epoch}, 전체: {total_steps}번")

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=config["epochs"],
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["batch_size"],
    gradient_accumulation_steps=config["grad_accum_steps"],
    learning_rate=config["lr"],
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(total_steps * 0.1)),
    weight_decay=0.0,
    bf16=config["bf16"],
    fp16=config["fp16"],
    gradient_checkpointing=config["gradient_checkpointing"],
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=collate,
)

eval_before = trainer.evaluate()["eval_loss"]
print(f"학습 전 평가 loss: {eval_before:.3f}")

In [ ]:
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

peak_gb = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else None
print(f"\n학습 시간: {elapsed / 60:.1f}분")
print(f"train loss 평균: {train_result.training_loss:.3f}")
if peak_gb is not None:
    print(f"학습 중 최대 VRAM 사용량: {peak_gb:.2f} GB")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(f.name == font for f in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = font
        plt.rcParams["axes.unicode_minus"] = False
        break

history = trainer.state.log_history
train_logs = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_logs = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*train_logs), label="train loss", marker=".")
if eval_logs:
    plt.plot(*zip(*eval_logs), label="eval loss", marker="o")
plt.xlabel("업데이트 스텝 (step)")
plt.ylabel("loss")
plt.title("LoRA 파인튜닝 loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. 학습 전후 비교: 어댑터 켜고 끄기

LoRA의 편리한 점: 원래 가중치는 그대로 있으므로 **`with model.disable_adapter():` 안에서는 베이스 모델로 돌아갑니다.**
2단계처럼 베이스 모델을 메모리에 한 번 더 불러올 필요가 없습니다.

In [ ]:
eval_after = trainer.evaluate()["eval_loss"]
print(f"평가 loss: 학습 전 {eval_before:.3f} → 학습 후 {eval_after:.3f}\n")

for q in test_questions:
    with model.disable_adapter():           # 어댑터 끔 = 베이스 모델
        base_answer = ask(model, q)
    print(f"Q: {q}")
    print(f"[어댑터 끔] {base_answer}")
    print(f"[어댑터 켬] {ask(model, q)}")
    print("-" * 60)

2단계 Full 파인튜닝과 비슷하게, 처음 보는 질문에도 냥이 말투로 짧게 답하고 `🐾` 뒤에 멈추는지 확인해 보세요.

### catastrophic forgetting 확인

어댑터를 끈 모델(= 베이스)과 켠 모델의 일반 문장 loss를 비교합니다. LoRA는 바꿀 수 있는 범위가 작아서 보통 Full 파인튜닝보다 원래 능력을 덜 잊어버립니다.

In [ ]:
general_texts = [
    "대한민국은 동아시아의 한반도 남부에 위치한 나라이다. 수도는 서울이다.",
    "광합성은 식물이 빛 에너지를 이용해 이산화탄소와 물로 포도당을 만드는 과정이다.",
    "The quick brown fox jumps over the lazy dog.",
]


@torch.no_grad()
def text_loss(model, texts):
    model.eval()
    losses = []
    for t in texts:
        batch = tokenizer(t, return_tensors="pt").to(model.device)
        losses.append(model(**batch, labels=batch["input_ids"]).loss.item())
    return sum(losses) / len(losses)


with model.disable_adapter():
    general_base = text_loss(model, general_texts)
general_lora = text_loss(model, general_texts)
print(f"일반 문장 loss: 베이스 {general_base:.3f} → LoRA 후 {general_lora:.3f}")

## 9. 어댑터만 저장하기

PEFT 모델에서 `save_pretrained`를 부르면 **LoRA 가중치(어댑터)만** 저장됩니다. 사용할 때는 베이스 모델을 불러온 다음 어댑터를 얹습니다.
2단계 Full 파인튜닝 저장 크기(약 2GB)와 비교해 보세요.

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / "adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

adapter_mb = (ADAPTER_DIR / "adapter_model.safetensors").stat().st_size / 1024**2
print(f"저장 위치: {ADAPTER_DIR.resolve()}")
print(f"어댑터 크기: {adapter_mb:.1f} MB")
for f in sorted(ADAPTER_DIR.iterdir()):
    print("  ", f.name)

print("\nadapter_config.json 일부:")
adapter_cfg = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
for k in ["base_model_name_or_path", "r", "lora_alpha", "target_modules"]:
    print(f"  {k}: {adapter_cfg[k]}")

`adapter_config.json`에 **어떤 베이스 모델에 붙이는 어댑터인지**(`base_model_name_or_path`)가 적혀 있습니다. 다른 베이스 모델에는 붙일 수 없습니다.

저장한 어댑터를 다시 불러와 봅시다. 베이스 모델을 새로 불러온 다음 `PeftModel.from_pretrained`로 어댑터를 얹습니다.

In [ ]:
from peft import PeftModel

del trainer, model
if device == "cuda":
    torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device, dtype=config["base_dtype"])
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
print(ask(model, "강아지는 왜 꼬리를 흔들어?"))

> 💡 어댑터를 베이스 가중치에 **합쳐서(merge)** 일반 모델처럼 쓰는 방법은 4단계에서 다룹니다.

## 10. Full 파인튜닝 vs LoRA 비교

이번 실행에서 측정한 값을 정리합니다. 2단계 칸은 2단계 노트북에서 출력된 값을 직접 적어 넣어 비교해 보세요.
(4단계에서는 두 모델을 같은 기준으로 자동 비교합니다.)

In [ ]:
metrics = {
    "trainable_params_M": round(n_trainable / 1e6, 2),
    "total_params_M": round((n_all - n_trainable) / 1e6, 1),
    "peak_vram_GB": round(peak_gb, 2) if peak_gb is not None else None,
    "train_minutes": round(elapsed / 60, 2),
    "eval_loss_before": round(eval_before, 3),
    "eval_loss_after": round(eval_after, 3),
    "general_loss_base": round(general_base, 3),
    "general_loss_after": round(general_lora, 3),
    "save_size_MB": round(adapter_mb, 1),
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

print(f"{'항목':<22}{'Full (2단계)':>16}{'LoRA (이번)':>16}")
print("-" * 54)
rows = [
    ("학습 파라미터", f"{metrics['total_params_M']}M", f"{metrics['trainable_params_M']}M"),
    ("최대 VRAM", "직접 기록", f"{peak_gb:.2f} GB" if peak_gb is not None else "(CPU)"),
    ("학습 시간", "직접 기록", f"{elapsed / 60:.1f}분"),
    ("평가 loss (후)", "직접 기록", f"{eval_after:.3f}"),
    ("일반 문장 loss (후)", "직접 기록", f"{general_lora:.3f}"),
    ("저장 크기", "약 2 GB", f"{adapter_mb:.0f} MB"),
]
for name, full, lora in rows:
    print(f"{name:<22}{full:>16}{lora:>16}")

**일반적으로 이런 결과가 나옵니다** (RTX 3060 12GB, Qwen2.5-0.5B 기준 대략적인 값):

| 항목 | Full 파인튜닝 | LoRA (r=16, 전체 Linear) |
|---|---|---|
| 학습 파라미터 | 494M (100%) | 약 8.8M (약 1.8%) |
| 모델 가중치 메모리 | fp32 약 2GB | bf16 약 1GB |
| gradient + AdamW 상태 | 약 6GB | 약 0.1GB |
| 최대 VRAM | 약 8~10GB | 약 3~5GB |
| 저장 크기 | 약 2GB | 약 35MB |
| 말투 학습 결과 | 잘 배움 | 거의 비슷하게 잘 배움 |
| 원래 능력 유지 | 조금 더 잊어버림 | 덜 잊어버림 |

> 📝 학습 **시간**은 크게 줄지 않습니다. 순전파/역전파는 여전히 모델 전체를 거쳐야 하기 때문입니다. LoRA가 아끼는 것은 주로 **메모리와 저장 공간**입니다.

**언제 무엇을 쓸까?**
- 메모리가 충분하고 큰 변화(새 언어, 대량의 도메인 지식)가 필요하면 → Full 파인튜닝
- 메모리가 부족하거나, 말투/형식/특정 작업처럼 좁은 과제이거나, 과제별로 어댑터를 여러 개 두고 바꿔 쓰고 싶으면 → **LoRA (실무에서 가장 많이 씀)**

## 11. (선택) 12GB에서 1.5B 모델을 LoRA로 학습하기

1단계에서 계산했듯이 Qwen2.5-1.5B를 Full 파인튜닝하려면 `1.5B × 16바이트 ≈ 24GB + activation`이 필요해서 **RTX 3060 12GB에는 들어가지 않습니다.**
LoRA라면 베이스 가중치 bf16 약 3GB + 작은 어댑터 + activation으로 충분히 들어갑니다.

아래 셀의 `RUN_1_5B = True`로 바꾸고 실행해 보세요. 모델(약 3GB)을 새로 내려받습니다.
메모리를 아끼기 위해 **gradient checkpointing**을 켭니다. Qwen2.5는 크기와 상관없이 같은 토크나이저를 쓰므로 위에서 만든 `tokenized` 데이터를 그대로 씁니다.

> ⚠️ CPU에서는 아주 오래 걸리므로(1시간 이상) GPU에서만 권장합니다.

In [ ]:
RUN_1_5B = False   # True로 바꾸면 실행
BIG_MODEL_NAME = "Qwen/Qwen2.5-1.5B"

if RUN_1_5B:
    if device == "cpu":
        print("⚠️ CPU에서는 매우 느립니다. 흐름 확인용으로 1 epoch만 돌립니다.")
    del model, base
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    big = AutoModelForCausalLM.from_pretrained(BIG_MODEL_NAME).to(device, dtype=config["base_dtype"])
    big.config.use_cache = False                       # gradient checkpointing과 함께 못 씀
    big = get_peft_model(big, LoraConfig(task_type="CAUSAL_LM", bias="none", **lora_settings))
    big.print_trainable_parameters()

    big_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "qwen1.5b"),
        num_train_epochs=config["epochs"] if device == "cuda" else 1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=config["lr"],
        lr_scheduler_type="cosine",
        warmup_steps=2,
        bf16=config["bf16"],
        fp16=config["fp16"],
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        seed=42,
    )
    big_trainer = Trainer(model=big, args=big_args, train_dataset=tokenized["train"], data_collator=collate)

    start = time.time()
    big_trainer.train()
    print(f"\n학습 시간: {(time.time() - start) / 60:.1f}분")
    if device == "cuda":
        print(f"최대 VRAM 사용량: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB  (Full이면 24GB 이상 필요)")

    big.config.use_cache = True
    for q in test_questions:
        print(f"\nQ: {q}\nA: {ask(big, q)}")
else:
    print("건너뜀. RUN_1_5B = True로 바꾸면 Qwen2.5-1.5B를 LoRA로 학습합니다.")

1.5B 모델은 0.5B보다 지식이 많아서 답변 내용이 더 정확하고 자연스러운 경우가 많습니다. **같은 GPU에서 더 큰 모델을 쓸 수 있게 해 주는 것**이 LoRA의 가장 큰 실용적 가치입니다.
더 큰 모델(3B, 7B)은 베이스 가중치를 4비트로 줄이는 **QLoRA**를 쓰면 12GB에 들어갑니다. 4단계에서 소개합니다.

## 12. 정리

| 개념 | 한 줄 요약 |
|---|---|
| LoRA | 원래 가중치 `W`는 얼리고 `W + B·A × (alpha/r)`의 작은 `A`, `B`만 학습 |
| `B = 0` 초기화 | 학습 시작 시점에 원래 모델과 똑같이 동작 |
| `r`, `lora_alpha` | 어댑터 크기와 배율. 보통 `r=8~64`, `alpha=r~2r` |
| `target_modules` | LoRA를 붙일 Linear. 어텐션 + MLP 전부에 붙이면 효과가 좋음 |
| 학습률 | Full보다 약 10배 큼 (`1e-4 ~ 3e-4`) |
| 메모리 절약 | gradient/옵티마이저 상태가 거의 없음 + 베이스를 bf16으로 |
| 저장 | 어댑터만 수십 MB. 사용 시 베이스 모델 + 어댑터 |
| `disable_adapter()` | 어댑터를 잠시 꺼서 베이스 모델로 비교 |

### 연습 문제

1. `r`을 4, 64로 바꿔 보세요. 학습 파라미터 수, 어댑터 크기, 결과가 어떻게 변하나요?
2. `target_modules`를 `["q_proj", "v_proj"]`만 남기고 학습해 보세요. (LoRA 논문의 원래 설정입니다.) 말투를 똑같이 잘 배우나요?
3. `lr`을 2단계와 같은 `3e-5`로 낮춰서 학습해 보세요. 왜 LoRA에는 큰 학습률이 필요할까요?
4. `config["gradient_checkpointing"] = True`로 바꾸고 최대 VRAM과 학습 시간을 비교해 보세요.
5. 2단계 결과와 나란히 놓고, 처음 보는 질문 몇 개를 더 만들어 Full과 LoRA의 답변을 비교해 보세요.

### 다음 단계

**04. 평가와 다음 단계**에서는 베이스, Full, LoRA 세 모델을 같은 기준으로 평가하고, 어댑터를 베이스 모델에 합치는(merge) 방법, 그리고 QLoRA와 SFT 데이터 만들기를 다룹니다.